In [1]:
import pandas as pd
import os
import re
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [2]:
data_dir = "../data/"

file_names = ['accepted_2007_to_2018Q4.csv', 'lc_2016_2017.csv', 'lc_loan.csv', 'lending_club_loan_two.csv', 'loan_4archive.csv']

dataframes = {file: pd.read_csv(os.path.join(data_dir, file), low_memory=False) for file in file_names}

for file, df in dataframes.items():
    print(f"File: {file}")
    print(df.info(), "\n")

File: accepted_2007_to_2018Q4.csv
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Columns: 151 entries, id to settlement_term
dtypes: float64(113), object(38)
memory usage: 2.5+ GB
None 

File: lc_2016_2017.csv
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 759338 entries, 0 to 759337
Data columns (total 72 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           759338 non-null  int64  
 1   member_id                    0 non-null       float64
 2   loan_amnt                    759338 non-null  int64  
 3   funded_amnt                  759338 non-null  int64  
 4   funded_amnt_inv              759338 non-null  float64
 5   term                         759338 non-null  object 
 6   int_rate                     759338 non-null  float64
 7   installment                  759338 non-null  float64
 8   grade                        759338 non-null  objec

In [3]:
for file, df in dataframes.items():
    print(f"Preview of {file}:")
    print(df.head(), "\n")

Preview of accepted_2007_to_2018Q4.csv:
         id  member_id  loan_amnt  funded_amnt  funded_amnt_inv        term  \
0  68407277        NaN     3600.0       3600.0           3600.0   36 months   
1  68355089        NaN    24700.0      24700.0          24700.0   36 months   
2  68341763        NaN    20000.0      20000.0          20000.0   60 months   
3  66310712        NaN    35000.0      35000.0          35000.0   60 months   
4  68476807        NaN    10400.0      10400.0          10400.0   60 months   

   int_rate  installment grade sub_grade  ... hardship_payoff_balance_amount  \
0     13.99       123.03     C        C4  ...                            NaN   
1     11.99       820.28     C        C1  ...                            NaN   
2     10.78       432.66     B        B4  ...                            NaN   
3     14.85       829.90     C        C5  ...                            NaN   
4     22.45       289.91     F        F1  ...                            NaN   

  ha

In [4]:
#identify the highest total missing value for each columns
for file, df in dataframes.items():
    missing_values = df.isnull().sum() / len(df) * 100
    print(f"Missing values in {file}:")
    print(missing_values[missing_values > 0].sort_values(ascending=False), "\n")


Missing values in accepted_2007_to_2018Q4.csv:
member_id                                     100.000000
orig_projected_additional_accrued_interest     99.617331
hardship_payoff_balance_amount                 99.517097
hardship_last_payment_amount                   99.517097
payment_plan_start_date                        99.517097
                                                 ...    
total_rec_int                                   0.001460
total_rec_prncp                                 0.001460
hardship_flag                                   0.001460
disbursement_method                             0.001460
debt_settlement_flag                            0.001460
Length: 150, dtype: float64 

Missing values in lc_2016_2017.csv:
member_id                      100.000000
desc                            99.997761
dti_joint                       95.522284
annual_inc_joint                95.522020
verification_status_joint       95.522020
mths_since_last_record          81.407621
mths_sin

In [5]:
columns_to_drop_high_na = [
    'member_id','orig_projected_additional_accrued_interest',
    'hardship_type', 'hardship_reason', 'hardship_status',
    'hardship_start_date', 'hardship_end_date', 'hardship_amount',
    'settlement_amount', 'settlement_percentage', 'settlement_term',
    'id', 'url', 'desc', 'title', 'initial_list_status', 
    'policy_code', 'application_type', 'out_prncp', 'out_prncp_inv', 
    'next_pymnt_d', 'payment_plan_start_date', 
    'hardship_flag', 'hardship_length', 'hardship_dpd', 
    'hardship_loan_status', 'hardship_payoff_balance_amount', 
    'hardship_last_payment_amount', 'debt_settlement_flag', 
    'debt_settlement_flag_date', 'settlement_status', 'settlement_date',
    'revol_bal_joint','purpose', 'open_acc_6m', 'open_act_il', 'open_il_12m', 
    'open_il_24m', 'mths_since_rcnt_il', 'total_bal_il', 'il_util',
    'max_bal_bc', 'all_util', 'total_rev_hi_lim', 'inq_fi', 'total_cu_tl', 
    'inq_last_12m','deferral_term', 'tot_hi_cred_lim', 'total_bal_ex_mort', 
    'total_bc_limit', 'total_il_high_credit_limit','mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op',
    'mo_sin_rcnt_tl', 'percent_bc_gt_75', 'pct_tl_nvr_dlq',
    'mths_since_last_delinq', 'mths_since_last_record', 'mths_since_last_major_derog',
    'mths_since_recent_bc_dlq', 'mths_since_recent_revol_delinq', 'open_rv_12m', 'open_rv_24m',
    'open_acc_6m', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'acc_open_past_24mths',
    'pub_rec_bankruptcies', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl',
    'num_bc_sats', 'num_bc_tl', 'num_rev_tl_bal_gt_0', 'num_tl_120dpd_2m', 'num_tl_30dpd',
    'num_tl_90g_dpd_24m', 'num_tl_op_past_12m','acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal',
    'chargeoff_within_12_mths','delinq_amnt', 'mths_since_recent_bc', 'mths_since_recent_inq', 'num_il_tl', 
    'num_op_rev_tl', 'num_rev_accts', 'num_sats', 'tax_liens'
]

columns_to_drop_redundant = [
    'funded_amnt_inv', 'out_prncp_inv', 'total_pymnt_inv', 
    'pymnt_plan', 'disbursement_method'
]

columns_to_drop_joint = [
    'annual_inc_joint', 'dti_joint', 'verification_status_joint'
]

columns_to_drop_sec_app = [
    'sec_app_earliest_cr_line', 'sec_app_inq_last_6mths', 'sec_app_mort_acc',
    'sec_app_open_acc', 'sec_app_revol_util', 'sec_app_open_act_il', 
    'sec_app_num_rev_accts', 'sec_app_chargeoff_within_12_mths',
    'sec_app_collections_12_mths_ex_med', 'sec_app_mths_since_last_major_derog','collections_12_mths_ex_med'
]

# Drop irrelevant columns
df.drop(columns=columns_to_drop_high_na + columns_to_drop_redundant + columns_to_drop_joint + columns_to_drop_sec_app, inplace=True)

# Fill missing numerical values with median
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns
df[numerical_cols] = df[numerical_cols].fillna(df[numerical_cols].median())

# Fill missing categorical values with mode
categorical_cols = df.select_dtypes(include=['object']).columns
df[categorical_cols] = df[categorical_cols].fillna(df[categorical_cols].mode().iloc[0])


In [6]:
df.to_csv(os.path.join("../processing_data","cleaned_loan_data.csv"), index=False)

In [7]:
cleaned_data = pd.read_csv(os.path.join("../processing_data","cleaned_loan_data.csv"))


In [8]:
print(cleaned_data.columns)
for item in cleaned_data.columns:
    print(item)

Index(['loan_amnt', 'funded_amnt', 'term', 'int_rate', 'installment', 'grade',
       'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc',
       'verification_status', 'issue_d', 'loan_status', 'zip_code',
       'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line',
       'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util',
       'total_acc', 'total_pymnt', 'total_rec_prncp', 'total_rec_int',
       'total_rec_late_fee', 'recoveries', 'collection_recovery_fee',
       'last_pymnt_d', 'last_pymnt_amnt', 'last_credit_pull_d', 'mort_acc'],
      dtype='object')
loan_amnt
funded_amnt
term
int_rate
installment
grade
sub_grade
emp_title
emp_length
home_ownership
annual_inc
verification_status
issue_d
loan_status
zip_code
addr_state
dti
delinq_2yrs
earliest_cr_line
inq_last_6mths
open_acc
pub_rec
revol_bal
revol_util
total_acc
total_pymnt
total_rec_prncp
total_rec_int
total_rec_late_fee
recoveries
collection_recovery_fee
last_pymnt_d
last_pymnt_amnt
last

In [9]:
print(cleaned_data.columns)


Index(['loan_amnt', 'funded_amnt', 'term', 'int_rate', 'installment', 'grade',
       'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc',
       'verification_status', 'issue_d', 'loan_status', 'zip_code',
       'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line',
       'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util',
       'total_acc', 'total_pymnt', 'total_rec_prncp', 'total_rec_int',
       'total_rec_late_fee', 'recoveries', 'collection_recovery_fee',
       'last_pymnt_d', 'last_pymnt_amnt', 'last_credit_pull_d', 'mort_acc'],
      dtype='object')


In [10]:
cleaned_data['earliest_cr_line'] = pd.to_datetime(
    cleaned_data['earliest_cr_line'], 
    format='%b-%Y',  # Assuming format like 'Jan-2008'
    errors='coerce'   # Handle invalid dates gracefully
)

cleaned_data['earliest_cr_line'] = pd.to_datetime(cleaned_data['earliest_cr_line'])
cleaned_data['credit_age_years'] = (pd.to_datetime('today') - cleaned_data['earliest_cr_line']).dt.days // 365
cleaned_data.drop(columns=['earliest_cr_line'], inplace=True)


In [11]:
cleaned_data.columns

Index(['loan_amnt', 'funded_amnt', 'term', 'int_rate', 'installment', 'grade',
       'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc',
       'verification_status', 'issue_d', 'loan_status', 'zip_code',
       'addr_state', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'open_acc',
       'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'total_pymnt',
       'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries',
       'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt',
       'last_credit_pull_d', 'mort_acc', 'credit_age_years'],
      dtype='object')

In [12]:
# Convert issue_d column separately
cleaned_data['issue_d'] = pd.to_datetime(cleaned_data['issue_d'], format='%b-%Y', errors='coerce')
cleaned_data['issue_year'] = cleaned_data['issue_d'].dt.year
cleaned_data.drop(columns=['issue_d'], inplace=True)

In [13]:
cleaned_data.columns

Index(['loan_amnt', 'funded_amnt', 'term', 'int_rate', 'installment', 'grade',
       'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc',
       'verification_status', 'loan_status', 'zip_code', 'addr_state', 'dti',
       'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal',
       'revol_util', 'total_acc', 'total_pymnt', 'total_rec_prncp',
       'total_rec_int', 'total_rec_late_fee', 'recoveries',
       'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt',
       'last_credit_pull_d', 'mort_acc', 'credit_age_years', 'issue_year'],
      dtype='object')

In [14]:
cleaned_data = pd.get_dummies(cleaned_data, columns=['grade', 'sub_grade', 'home_ownership', 'verification_status'], drop_first=True)


In [15]:
# Save cleaned data
cleaned_data.to_csv(os.path.join("../processing_data", "second_cleaned_loan_data.csv"), index=False)



In [16]:
second_time_cleaned_data = pd.read_csv(os.path.join("../processing_data","second_cleaned_loan_data.csv"))
print(second_time_cleaned_data.columns)

Index(['loan_amnt', 'funded_amnt', 'term', 'int_rate', 'installment',
       'emp_title', 'emp_length', 'annual_inc', 'loan_status', 'zip_code',
       'addr_state', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'open_acc',
       'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'total_pymnt',
       'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries',
       'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt',
       'last_credit_pull_d', 'mort_acc', 'credit_age_years', 'issue_year',
       'grade_B', 'grade_C', 'grade_D', 'grade_E', 'grade_F', 'grade_G',
       'sub_grade_A2', 'sub_grade_A3', 'sub_grade_A4', 'sub_grade_A5',
       'sub_grade_B1', 'sub_grade_B2', 'sub_grade_B3', 'sub_grade_B4',
       'sub_grade_B5', 'sub_grade_C1', 'sub_grade_C2', 'sub_grade_C3',
       'sub_grade_C4', 'sub_grade_C5', 'sub_grade_D1', 'sub_grade_D2',
       'sub_grade_D3', 'sub_grade_D4', 'sub_grade_D5', 'sub_grade_E1',
       'sub_grade_E2', 'sub_grade_E3', 'sub_grade_E4', 'su

In [17]:
second_time_cleaned_data

,loan_amnt,funded_amnt,term,int_rate,installment,emp_title,emp_length,annual_inc,loan_status,zip_code,...,sub_grade_G3,sub_grade_G4,sub_grade_G5,home_ownership_MORTGAGE,home_ownership_NONE,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT,verification_status_Source Verified,verification_status_Verified
0,2500,2500,36 months,13.56,84.92,Chef,10+ years,55000.0,Current,109xx,...,False,False,False,False,False,False,False,True,False,False
1,30000,30000,60 months,18.94,777.23,Postmaster,10+ years,90000.0,Current,713xx,...,False,False,False,True,False,False,False,False,True,False
2,5000,5000,36 months,17.97,180.69,Administrative,6 years,59280.0,Current,490xx,...,False,False,False,True,False,False,False,False,True,False
3,4000,4000,36 months,18.94,146.51,IT Supervisor,10+ years,92000.0,Current,985xx,...,False,False,False,True,False,False,False,False,True,False
4,30000,30000,60 months,16.14,731.78,Mechanic,10+ years,57250.0,Current,212xx,...,False,False,False,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2260663,12000,12000,60 months,14.08,279.72,house keeper,10+ years,58000.0,Current,054xx,...,False,False,False,True,False,False,False,False,False,False
2260664,12000,12000,60 months,25.82,358.01,Skilled Labor,< 1 year,30000.0,Fully Paid,971xx,...,False,False,False,True,False,False,False,False,False,False
2260665,10000,10000,36 months,11.99,332.10,Teacher,10+ years,64000.0,Current,603xx,...,False,False,False,False,False,False,True,False,True,False
2260666,12000,12000,60 months,21.45,327.69,Teacher,10+ years,60000.0,Current,996xx,...,False,False,False,False,False,False,False,True,False,False


In [18]:
second_time_cleaned_data[['loan_amnt', 'int_rate', 'dti']]

,loan_amnt,int_rate,dti
0,2500,13.56,18.24
1,30000,18.94,26.52
2,5000,17.97,10.51
3,4000,18.94,16.74
4,30000,16.14,26.35
...,...,...,...
2260663,12000,14.08,20.88
2260664,12000,25.82,19.28
2260665,10000,11.99,12.96
2260666,12000,21.45,30.82


In [19]:
scaler = MinMaxScaler()
second_time_cleaned_data[['loan_amnt', 'int_rate', 'dti']] = scaler.fit_transform(second_time_cleaned_data[['loan_amnt', 'int_rate', 'dti']])


In [20]:
second_time_cleaned_data[['loan_amnt', 'int_rate', 'dti']]

,loan_amnt,int_rate,dti
0,0.050633,0.321262,0.01924
1,0.746835,0.530763,0.02752
2,0.113924,0.492991,0.01151
3,0.088608,0.530763,0.01774
4,0.746835,0.421729,0.02735
...,...,...,...
2260663,0.291139,0.341511,0.02188
2260664,0.291139,0.798676,0.02028
2260665,0.240506,0.260125,0.01396
2260666,0.291139,0.628505,0.03182


In [21]:
print(f"Total unique job titles: {second_time_cleaned_data['emp_title'].nunique()}")


Total unique job titles: 512694


In [22]:
second_time_cleaned_data['emp_title'] = second_time_cleaned_data['emp_title'].str.lower().str.strip()  # Convert to lowercase and strip whitespace


In [23]:
second_time_cleaned_data['emp_title']

0                    chef
1              postmaster
2          administrative
3           it supervisor
4                mechanic
                ...      
2260663      house keeper
2260664     skilled labor
2260665           teacher
2260666           teacher
2260667        babysitter
Name: emp_title, Length: 2260668, dtype: object

In [24]:
job_title_counts = second_time_cleaned_data['emp_title'].value_counts()
print(job_title_counts.head(20))

# Save all job titles and their counts to a CSV file
job_title_counts.to_csv(os.path.join("../processing_data", "unique_job_titles.csv"), header=['count'])



emp_title
teacher               215429
manager                45852
owner                  33591
registered nurse       23354
supervisor             22306
driver                 22267
sales                  18984
rn                     17196
office manager         14231
project manager        13842
general manager        13317
truck driver           12797
director               10595
president               9826
engineer                8978
sales manager           8532
operations manager      8183
police officer          7675
vice president          7625
technician              7437
Name: count, dtype: int64


In [25]:
def categorize_job_title(title):
    if pd.isna(title):
        return 'unknown'
    
    lower_title = title.lower().strip()
    
    # Software/IT related
    if re.search(r'software|developer|programmer|web|frontend|backend|full stack|fullstack|devops|systems analyst|data scientist|data engineer', lower_title):
        return 'software_it'
    
    # Other Engineering
    if re.search(r'engineer', lower_title):
        if re.search(r'mechanical engineer', lower_title):
            return 'mechanical_engineering'
        elif re.search(r'civil engineer', lower_title):
            return 'civil_engineering'
        elif re.search(r'electrical engineer', lower_title):
            return 'electrical_engineering'
        elif re.search(r'chemical engineer', lower_title):
            return 'chemical_engineering'
        else:
            return 'other_engineering'
            
    # Manager group
    if re.search(r'\manager\', lower_title):
        return 'manager'
        
    # Nursing group
    if re.search(r'\rn\|\lpn\|\cna\|nurse|nursing', lower_title):
        return 'nursing'
        
    # Teacher group
    if re.search(r'\teacher\|instructor|professor|educator', lower_title):
        return 'education'
        
    # Sales group
    if re.search(r'sales|account executive|business development', lower_title):
        return 'sales'
        
    # Owner/Entrepreneur group
    if re.search(r'\owner\|entrepreneur|founder|ceo|president', lower_title):
        return 'executive_owner'
        
    # Supervisor group
    if re.search(r'\supervisor\|\team lead\|coordinator', lower_title):
        return 'supervisor'
        
    # Administrative
    if re.search(r'admin|secretary|clerk|receptionist|office assistant', lower_title):
        return 'administrative'
        
    # Medical (non-nursing)
    if re.search(r'doctor|physician|surgeon|dentist|pharmacist|therapist', lower_title):
        return 'medical'
        
    # Finance
    if re.search(r'accountant|financial|banker|analyst|broker', lower_title):
        return 'finance'
        
    # Designer
    if re.search(r'designer|architect|artist|creative|graphic', lower_title):
        return 'design_creative'
    
    return 'other'

# Apply categorization
second_time_cleaned_data['job_category'] = second_time_cleaned_data['emp_title'].apply(categorize_job_title)

# Show the distribution
category_counts = second_time_cleaned_data['job_category'].value_counts()
print("Job Category Distribution:")
print(category_counts)

# Show some example titles for each category
print("Example titles for each category:")
for category in category_counts.index:
    examples = second_time_cleaned_data[second_time_cleaned_data['job_category'] == category]['emp_title'].head(3).tolist()
    print(f"{category}:")
    print(examples)



Job Category Distribution:
job_category
other                     1777313
sales                       83795
finance                     77987
administrative              74706
nursing                     51495
other_engineering           48512
executive_owner             32931
supervisor                  31487
software_it                 28127
medical                     22431
design_creative             14842
education                   13938
electrical_engineering       1175
mechanical_engineering       1046
civil_engineering             774
chemical_engineering          109
Name: count, dtype: int64
Example titles for each category:
other:
['chef', 'postmaster', 'it supervisor']
sales:
['sales', 'senior area sales manager', 'sales vp']
finance:
['financial relationship associate', 'banker', 'business analyst']
administrative:
['administrative', 'administrative assistant', 'receptionist']
nursing:
['neonatal nurse practitioner', 'nursing supervisor', 'nurse technician']
other_enginee

In [26]:
second_time_cleaned_data['job_category'] = second_time_cleaned_data['emp_title'].apply(categorize_job_title)

In [27]:
second_time_cleaned_data.drop(columns=['emp_title', 'loan_status'], inplace=True)

In [28]:
second_time_cleaned_data['emp_length']

0          10+ years
1          10+ years
2            6 years
3          10+ years
4          10+ years
             ...    
2260663    10+ years
2260664     < 1 year
2260665    10+ years
2260666    10+ years
2260667      3 years
Name: emp_length, Length: 2260668, dtype: object

In [29]:
def convert_emp_length_to_numeric(length):
    if pd.isna(length) or length == 'n/a':
        return np.nan  # Handle missing values with NaN
    if '<' in length:  # Handle less than 1 year
        return 0
    numeric_part = int(length.split()[0].replace('+', ''))  # Extract the numeric value
    return min(numeric_part, 10)  # Cap the maximum value at 10

# Apply the function to convert the column
second_time_cleaned_data['emp_length'] = second_time_cleaned_data['emp_length'].apply(convert_emp_length_to_numeric)


In [30]:
second_time_cleaned_data['emp_length']

0          10
1          10
2           6
3          10
4          10
           ..
2260663    10
2260664     0
2260665    10
2260666    10
2260667     3
Name: emp_length, Length: 2260668, dtype: int64

In [31]:

# Function to extract the numeric value from the term string
def extract_term_value(term):
    match = re.search(r'\d+', term)
    return int(match.group()) if match else None

In [32]:
second_time_cleaned_data['term'] = second_time_cleaned_data['term'].apply(extract_term_value)

In [33]:
second_time_cleaned_data

,loan_amnt,funded_amnt,term,int_rate,installment,emp_length,annual_inc,zip_code,addr_state,dti,...,sub_grade_G4,sub_grade_G5,home_ownership_MORTGAGE,home_ownership_NONE,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT,verification_status_Source Verified,verification_status_Verified,job_category
0,0.050633,2500,36,0.321262,84.92,10,55000.0,109xx,NY,0.01924,...,False,False,False,False,False,False,True,False,False,other
1,0.746835,30000,60,0.530763,777.23,10,90000.0,713xx,LA,0.02752,...,False,False,True,False,False,False,False,True,False,other
2,0.113924,5000,36,0.492991,180.69,6,59280.0,490xx,MI,0.01151,...,False,False,True,False,False,False,False,True,False,administrative
3,0.088608,4000,36,0.530763,146.51,10,92000.0,985xx,WA,0.01774,...,False,False,True,False,False,False,False,True,False,other
4,0.746835,30000,60,0.421729,731.78,10,57250.0,212xx,MD,0.02735,...,False,False,True,False,False,False,False,False,False,other
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2260663,0.291139,12000,60,0.341511,279.72,10,58000.0,054xx,VT,0.02188,...,False,False,True,False,False,False,False,False,False,other
2260664,0.291139,12000,60,0.798676,358.01,0,30000.0,971xx,OR,0.02028,...,False,False,True,False,False,False,False,False,False,other
2260665,0.240506,10000,36,0.260125,332.10,10,64000.0,603xx,IL,0.01396,...,False,False,False,False,False,True,False,True,False,other
2260666,0.291139,12000,60,0.628505,327.69,10,60000.0,996xx,AK,0.03182,...,False,False,False,False,False,False,True,False,False,other


In [34]:
second_time_cleaned_data.columns

Index(['loan_amnt', 'funded_amnt', 'term', 'int_rate', 'installment',
       'emp_length', 'annual_inc', 'zip_code', 'addr_state', 'dti',
       'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal',
       'revol_util', 'total_acc', 'total_pymnt', 'total_rec_prncp',
       'total_rec_int', 'total_rec_late_fee', 'recoveries',
       'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt',
       'last_credit_pull_d', 'mort_acc', 'credit_age_years', 'issue_year',
       'grade_B', 'grade_C', 'grade_D', 'grade_E', 'grade_F', 'grade_G',
       'sub_grade_A2', 'sub_grade_A3', 'sub_grade_A4', 'sub_grade_A5',
       'sub_grade_B1', 'sub_grade_B2', 'sub_grade_B3', 'sub_grade_B4',
       'sub_grade_B5', 'sub_grade_C1', 'sub_grade_C2', 'sub_grade_C3',
       'sub_grade_C4', 'sub_grade_C5', 'sub_grade_D1', 'sub_grade_D2',
       'sub_grade_D3', 'sub_grade_D4', 'sub_grade_D5', 'sub_grade_E1',
       'sub_grade_E2', 'sub_grade_E3', 'sub_grade_E4', 'sub_grade_E5',
       'sub_gra

In [35]:
second_time_cleaned_data['addr_state']

0          NY
1          LA
2          MI
3          WA
4          MD
           ..
2260663    VT
2260664    OR
2260665    IL
2260666    AK
2260667    NY
Name: addr_state, Length: 2260668, dtype: object

In [36]:
unique_zip_prefixes = second_time_cleaned_data['zip_code'].str[:3]

print("Unique ZIP prefixes to query:", unique_zip_prefixes)

Unique ZIP prefixes to query: 0          109
1          713
2          490
3          985
4          212
          ... 
2260663    054
2260664    971
2260665    603
2260666    996
2260667    112
Name: zip_code, Length: 2260668, dtype: object


In [37]:
second_time_cleaned_data['zip_code'] = unique_zip_prefixes

In [38]:
second_time_cleaned_data["last_pymnt_d"] = pd.to_datetime(second_time_cleaned_data["last_pymnt_d"], format="%b-%Y", errors="coerce")
second_time_cleaned_data["last_credit_pull_d"] = pd.to_datetime(second_time_cleaned_data["last_credit_pull_d"], format="%b-%Y", errors="coerce")

# Convert to YYYYMM numeric format (e.g., Feb-2019 → 201902)
second_time_cleaned_data["last_pymnt_d"] = second_time_cleaned_data["last_pymnt_d"].dt.year * 100 + second_time_cleaned_data["last_pymnt_d"].dt.month
second_time_cleaned_data["last_credit_pull_d"] = second_time_cleaned_data["last_credit_pull_d"].dt.year * 100 + second_time_cleaned_data["last_credit_pull_d"].dt.month

# Fill missing values with earliest valid year (e.g., 201901)
second_time_cleaned_data["last_pymnt_d"].fillna(201901, inplace=True)
second_time_cleaned_data["last_credit_pull_d"].fillna(201901, inplace=True)

C:\Users\mingx\AppData\Local\Temp\ipykernel_16752\4111066070.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  second_time_cleaned_data["last_pymnt_d"].fillna(201901, inplace=True)
C:\Users\mingx\AppData\Local\Temp\ipykernel_16752\4111066070.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behav

In [39]:
# addr_state_encoded = pd.get_dummies(second_time_cleaned_data["addr_state"], prefix="addr_state")

# second_time_cleaned_data = pd.concat([second_time_cleaned_data, addr_state_encoded], axis=1)

second_time_cleaned_data = pd.get_dummies(second_time_cleaned_data, columns=["addr_state"], drop_first=True)

In [40]:
second_time_cleaned_data = pd.get_dummies(second_time_cleaned_data, columns=["job_category"], drop_first=True)

In [41]:
second_time_cleaned_data

,loan_amnt,funded_amnt,term,int_rate,installment,emp_length,annual_inc,zip_code,dti,delinq_2yrs,...,job_category_executive_owner,job_category_finance,job_category_mechanical_engineering,job_category_medical,job_category_nursing,job_category_other,job_category_other_engineering,job_category_sales,job_category_software_it,job_category_supervisor
0,0.050633,2500,36,0.321262,84.92,10,55000.0,109,0.01924,0.0,...,False,False,False,False,False,True,False,False,False,False
1,0.746835,30000,60,0.530763,777.23,10,90000.0,713,0.02752,0.0,...,False,False,False,False,False,True,False,False,False,False
2,0.113924,5000,36,0.492991,180.69,6,59280.0,490,0.01151,0.0,...,False,False,False,False,False,False,False,False,False,False
3,0.088608,4000,36,0.530763,146.51,10,92000.0,985,0.01774,0.0,...,False,False,False,False,False,True,False,False,False,False
4,0.746835,30000,60,0.421729,731.78,10,57250.0,212,0.02735,0.0,...,False,False,False,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2260663,0.291139,12000,60,0.341511,279.72,10,58000.0,054,0.02188,0.0,...,False,False,False,False,False,True,False,False,False,False
2260664,0.291139,12000,60,0.798676,358.01,0,30000.0,971,0.02028,3.0,...,False,False,False,False,False,True,False,False,False,False
2260665,0.240506,10000,36,0.260125,332.10,10,64000.0,603,0.01396,0.0,...,False,False,False,False,False,True,False,False,False,False
2260666,0.291139,12000,60,0.628505,327.69,10,60000.0,996,0.03182,2.0,...,False,False,False,False,False,True,False,False,False,False


In [42]:
second_time_cleaned_data.to_csv("../cleaned_data/final_data_cleaned.csv", index=False)

In [43]:
final_data = pd.read_csv("../cleaned_data/final_data_cleaned.csv")

In [44]:
final_data

,loan_amnt,funded_amnt,term,int_rate,installment,emp_length,annual_inc,zip_code,dti,delinq_2yrs,...,job_category_executive_owner,job_category_finance,job_category_mechanical_engineering,job_category_medical,job_category_nursing,job_category_other,job_category_other_engineering,job_category_sales,job_category_software_it,job_category_supervisor
0,0.050633,2500,36,0.321262,84.92,10,55000.0,109,0.01924,0.0,...,False,False,False,False,False,True,False,False,False,False
1,0.746835,30000,60,0.530763,777.23,10,90000.0,713,0.02752,0.0,...,False,False,False,False,False,True,False,False,False,False
2,0.113924,5000,36,0.492991,180.69,6,59280.0,490,0.01151,0.0,...,False,False,False,False,False,False,False,False,False,False
3,0.088608,4000,36,0.530763,146.51,10,92000.0,985,0.01774,0.0,...,False,False,False,False,False,True,False,False,False,False
4,0.746835,30000,60,0.421729,731.78,10,57250.0,212,0.02735,0.0,...,False,False,False,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2260663,0.291139,12000,60,0.341511,279.72,10,58000.0,54,0.02188,0.0,...,False,False,False,False,False,True,False,False,False,False
2260664,0.291139,12000,60,0.798676,358.01,0,30000.0,971,0.02028,3.0,...,False,False,False,False,False,True,False,False,False,False
2260665,0.240506,10000,36,0.260125,332.10,10,64000.0,603,0.01396,0.0,...,False,False,False,False,False,True,False,False,False,False
2260666,0.291139,12000,60,0.628505,327.69,10,60000.0,996,0.03182,2.0,...,False,False,False,False,False,True,False,False,False,False


In [45]:
final_data.columns

Index(['loan_amnt', 'funded_amnt', 'term', 'int_rate', 'installment',
       'emp_length', 'annual_inc', 'zip_code', 'dti', 'delinq_2yrs',
       ...
       'job_category_executive_owner', 'job_category_finance',
       'job_category_mechanical_engineering', 'job_category_medical',
       'job_category_nursing', 'job_category_other',
       'job_category_other_engineering', 'job_category_sales',
       'job_category_software_it', 'job_category_supervisor'],
      dtype='object', length=140)